In [0]:
import urllib.request
import json

url = "https://data.cityofchicago.org/resource/wrvz-psew.json?$limit=1000"

with urllib.request.urlopen(url) as response:
    raw = json.loads(response.read().decode())

print(f"Records fetched: {len(raw)}")
print(f"Sample record:")
for key, value in raw[0].items():
    print(f"  {key}: {value}")

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, round, when, hour, dayofweek, to_timestamp, lit, current_timestamp
from pyspark.sql.types import(
    StructType, StructField, StringType, DoubleType, IntegerType
)

def safe_float(val):
    try:
        return float(val)
    except:
        return None
    
def safe_int(val):
    try:
        return int(val)
    except:
        return None
    
rows = []
for r in raw:
    rows.append(Row(
        trip_id = r.get("trip_id", None),
        taxi_id = r.get("taxi_id", None),
        trip_start = r.get("trip_start_timestamp", None),
        trip_end = r.get("trip_end_timestamp", None),
        trip_seconds = safe_int(r.get("trip_seconds", 0)),
        trip_miles = safe_float(r.get("trip_miles",0)),
        fare = safe_float(r.get("fare",0)),
        tips = safe_float(r.get("tips", 0)),
        tolls = safe_float(r.get("tolls",0)),
        trip_total = safe_float(r.get("trip_total", 0)),
        payment_type = r.get("payment_type", None),
        company = r.get("company", None) 
    ))

df_raw = spark.createDataFrame(rows)
print(f"Rows loaded: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5)

In [0]:
import urllib.request
import json
from pyspark.sql.functions import col, round, when, lit, current_timestamp, hour, to_timestamp
from pyspark.sql.types import DateType

run_date = "2024-01-01"

print(f"Run date: {run_date}")

In [0]:
df_bronze = df_raw \
    .withColumn("ingested_at", current_timestamp()) \
    .withColumn("run_date",    lit(run_date))

row_count    = df_bronze.count()
null_fare = df_bronze.filter(col("fare").isNull()).count()

print(f"Row count:        {row_count}")
print(f"Null amounts:     {null_fare}")

if row_count == 0:
    raise Exception("BRONZE FAILED: No rows to write — aborting pipeline")

if null_fare > 0:
    print(f"WARNING: {null_fare} rows have null amounts — fares")

print("Validation passed — proceeding to write")
spark.sql("DROP TABLE IF EXISTS taxi_bronze")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_bronze")

print(f"taxi_bronze written — {spark.read.table('taxi_bronze').count()} rows")